In [1]:
# ── ONE-TIME SETUP: Download winutils for Windows ────────────
import urllib.request, os, stat
WINUTILS_DIR = r"C:\hadoop\bin"
os.makedirs(WINUTILS_DIR, exist_ok=True)
winutils_path = os.path.join(WINUTILS_DIR, "winutils.exe")
hadoop_dll    = os.path.join(WINUTILS_DIR, "hadoop.dll")
if not os.path.exists(winutils_path):
    print("Downloading winutils.exe …")
    urllib.request.urlretrieve(
        "https://github.com/cdarlint/winutils/raw/master/hadoop-3.3.5/bin/winutils.exe",
        winutils_path
    )
    print("  ✓ winutils.exe downloaded")
else:
    print("  ✓ winutils.exe already exists")

if not os.path.exists(hadoop_dll):
    print("Downloading hadoop.dll …")
    urllib.request.urlretrieve(
        "https://github.com/cdarlint/winutils/raw/master/hadoop-3.3.5/bin/hadoop.dll",
        hadoop_dll
    )
    print("  ✓ hadoop.dll downloaded")
else:
    print("  ✓ hadoop.dll already exists")
# Set environment variables for this session
os.environ["HADOOP_HOME"]    = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
# Verify
print(f"\nHADOOP_HOME = {os.environ['HADOOP_HOME']}")
print(f"winutils exists: {os.path.exists(winutils_path)}")
print("✓ Hadoop setup complete")

  ✓ winutils.exe downloaded
  ✓ hadoop.dll downloaded

HADOOP_HOME = C:\hadoop
winutils exists: True
✓ Hadoop setup complete


## Data Ingestion

In [2]:
# ── Cell 1: Environment setup — MUST run before everything else ──
import os

# Set HADOOP_HOME before Spark starts — order matters on Windows
os.environ["HADOOP_HOME"]     = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"]            = r"C:\hadoop\bin;" + os.environ.get("PATH", "")

print(f"HADOOP_HOME set to: {os.environ['HADOOP_HOME']}")

# Verify winutils exists
winutils = r"C:\hadoop\bin\winutils.exe"
if os.path.exists(winutils):
    print(f"✓ winutils.exe found")
else:
    raise FileNotFoundError(
        "winutils.exe not found at C:\\hadoop\\bin\\winutils.exe\n"
        "Run the setup cell above first!"
    )

HADOOP_HOME set to: C:\hadoop
✓ winutils.exe found


In [3]:
# ── Cell 2: Imports ──────────────────────────────────────────
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import time, gzip, shutil

In [4]:
# ── Cell 3: Paths ────────────────────────────────────────────
SOURCE_DIR = r"C:\Users\PC\Downloads\bag+of+words"
DATA_DIR   = "./data/raw"
PAR_DIR    = "./data/parquet"

for d in [DATA_DIR, "./data/processed", PAR_DIR, "./models", "./outputs"]:
    os.makedirs(d, exist_ok=True)
    print(f"  ✓ {d}")

  ✓ ./data/raw
  ✓ ./data/processed
  ✓ ./data/parquet
  ✓ ./models
  ✓ ./outputs


In [5]:
# ── Cell 4: Extract .gz files ────────────────────────────────
GZ_FILES    = ["docword.kos.txt.gz", "docword.nips.txt.gz"]
PLAIN_FILES = ["vocab.kos.txt", "vocab.nips.txt"]

for gz_name in GZ_FILES:
    txt_name = gz_name.replace(".gz", "")
    src = os.path.join(SOURCE_DIR, gz_name)
    dst = os.path.join(DATA_DIR, txt_name)
    if os.path.exists(dst):
        print(f"  ✓ {txt_name} already extracted")
    else:
        print(f"  Extracting {gz_name} …", end=" ")
        with gzip.open(src, "rb") as f_in, open(dst, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
        print(f"done ({os.path.getsize(dst)/1024/1024:.1f} MB)")

for fname in PLAIN_FILES:
    src = os.path.join(SOURCE_DIR, fname)
    dst = os.path.join(DATA_DIR, fname)
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f"  ✓ Copied {fname}")
    else:
        print(f"  ✓ {fname} already in place")

  ✓ docword.kos.txt already extracted
  ✓ docword.nips.txt already extracted
  ✓ vocab.kos.txt already in place
  ✓ vocab.nips.txt already in place


In [6]:
# ── Cell 5: SparkSession ─────────────────────────────────────
spark = SparkSession.builder \
    .appName("BagOfWords_Ingestion") \
    .master("local[1]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.hadoop.hadoop.home.dir", r"C:\hadoop") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"Spark UI      : {spark.sparkContext.uiWebUrl}")

Spark version : 4.1.1
Spark UI      : http://DESKTOP-6826D4K:4043


In [7]:
# ── Cell 6: Read headers ─────────────────────────────────────
def read_bow_header(filepath):
    with open(filepath, "r") as f:
        D   = int(f.readline().strip())
        W   = int(f.readline().strip())
        NNZ = int(f.readline().strip())
    return D, W, NNZ

kos_path  = os.path.join(DATA_DIR, "docword.kos.txt")
nips_path = os.path.join(DATA_DIR, "docword.nips.txt")

D_kos,  W_kos,  NNZ_kos  = read_bow_header(kos_path)
D_nips, W_nips, NNZ_nips = read_bow_header(nips_path)

print(f"KOS  — D={D_kos:,}  W={W_kos:,}  NNZ={NNZ_kos:,}")
print(f"NIPS — D={D_nips:,}  W={W_nips:,}  NNZ={NNZ_nips:,}")

KOS  — D=3,430  W=6,906  NNZ=353,160
NIPS — D=1,500  W=12,419  NNZ=746,316


In [8]:
# ── Cell 7: pandas → PyArrow → Parquet (no Spark workers) ────
def bow_to_parquet(txt_path, parquet_path, skip_lines=3):
    print(f"  Reading {os.path.basename(txt_path)} …")
    t0 = time.time()
    pdf = pd.read_csv(
        txt_path, sep=r"\s+", skiprows=skip_lines,
        header=None, names=["docID", "wordID", "count"],
        dtype={"docID": "int32", "wordID": "int32", "count": "int32"},
        engine="python"
    )
    print(f"    {len(pdf):,} rows | {pdf.memory_usage(deep=True).sum()/1024/1024:.1f} MB | {time.time()-t0:.2f}s")
    table = pa.Table.from_pandas(pdf, preserve_index=False)
    pq.write_table(table, parquet_path, compression="snappy", row_group_size=50000)
    print(f"    Saved to {parquet_path} ({os.path.getsize(parquet_path)/1024/1024:.2f} MB)")
    return len(pdf)

def vocab_to_parquet(vocab_path, parquet_path):
    print(f"  Reading {os.path.basename(vocab_path)} …", end=" ")
    pdf = pd.read_csv(vocab_path, header=None, names=["word"], dtype=str, engine="python")
    pdf.insert(0, "wordID", range(1, len(pdf) + 1))
    pdf["wordID"] = pdf["wordID"].astype("int32")
    pq.write_table(pa.Table.from_pandas(pdf, preserve_index=False), parquet_path, compression="snappy")
    print(f"{len(pdf):,} words saved")
    return len(pdf)

kos_parquet_path   = os.path.join(PAR_DIR, "kos_docword.parquet")
nips_parquet_path  = os.path.join(PAR_DIR, "nips_docword.parquet")
kosv_parquet_path  = os.path.join(PAR_DIR, "kos_vocab.parquet")
nipsv_parquet_path = os.path.join(PAR_DIR, "nips_vocab.parquet")

print("Step 1 — Writing Parquet via PyArrow (no Spark workers):")
kos_count  = bow_to_parquet(kos_path,  kos_parquet_path)
nips_count = bow_to_parquet(nips_path, nips_parquet_path)
vocab_to_parquet(os.path.join(DATA_DIR, "vocab.kos.txt"),  kosv_parquet_path)
vocab_to_parquet(os.path.join(DATA_DIR, "vocab.nips.txt"), nipsv_parquet_path)
print("✓ All Parquet files written")

Step 1 — Writing Parquet via PyArrow (no Spark workers):
  Reading docword.kos.txt …
    353,160 rows | 4.0 MB | 1.27s
    Saved to ./data/parquet\kos_docword.parquet (0.85 MB)
  Reading docword.nips.txt …
    746,316 rows | 8.5 MB | 2.57s
    Saved to ./data/parquet\nips_docword.parquet (2.08 MB)
  Reading vocab.kos.txt … 6,906 words saved
  Reading vocab.nips.txt … 12,419 words saved
✓ All Parquet files written


In [9]:
# ── Cell 8: Spark reads Parquet (JVM-native) ─────────────────
bow_schema = StructType([
    StructField("docID",  IntegerType(), False),
    StructField("wordID", IntegerType(), False),
    StructField("count",  IntegerType(), False),
])
vocab_schema = StructType([
    StructField("wordID", IntegerType(), False),
    StructField("word",   StringType(),  False),
])

print("Step 2 — Spark loading from Parquet:")
t0     = time.time()
kos_df = spark.read.schema(bow_schema).parquet(kos_parquet_path)
kos_df.cache()
rb_kos = kos_df.count()
print(f"  KOS : {rb_kos:,} rows in {time.time()-t0:.2f}s")
kos_df.show(5)

t0      = time.time()
nips_df = spark.read.schema(bow_schema).parquet(nips_parquet_path)
nips_df.cache()
rb_nips = nips_df.count()
print(f"  NIPS: {rb_nips:,} rows in {time.time()-t0:.2f}s")

kos_vocab_df  = spark.read.schema(vocab_schema).parquet(kosv_parquet_path)
nips_vocab_df = spark.read.schema(vocab_schema).parquet(nipsv_parquet_path)
print(f"  KOS  vocab: {kos_vocab_df.count():,} words")
print(f"  NIPS vocab: {nips_vocab_df.count():,} words")

assert rb_kos == NNZ_kos,   f"KOS  NNZ mismatch: {rb_kos} vs {NNZ_kos}"
assert rb_nips == NNZ_nips, f"NIPS NNZ mismatch: {rb_nips} vs {NNZ_nips}"
print("✓ Row counts match headers")

Step 2 — Spark loading from Parquet:
  KOS : 353,160 rows in 3.39s
+-----+------+-----+
|docID|wordID|count|
+-----+------+-----+
|    1|    61|    2|
|    1|    76|    1|
|    1|    89|    1|
|    1|   211|    1|
|    1|   296|    1|
+-----+------+-----+
only showing top 5 rows
  NIPS: 746,316 rows in 0.52s
  KOS  vocab: 6,906 words
  NIPS vocab: 12,419 words
✓ Row counts match headers


In [10]:
# ── Cell 9: Data Validation ──────────────────────────────────
print("=" * 55)
print("DATA VALIDATION — KOS")
print("=" * 55)

print("\n1. Null Check:")
for col in ["docID", "wordID", "count"]:
    n = kos_df.filter(F.col(col).isNull()).count()
    print(f"   {'✓' if n==0 else '⚠'} {col}: {n} nulls")

print("\n2. Value Range Check:")
stats = kos_df.agg(
    F.min("docID").alias("min_doc"),   F.max("docID").alias("max_doc"),
    F.min("wordID").alias("min_word"), F.max("wordID").alias("max_word"),
    F.min("count").alias("min_cnt"),   F.max("count").alias("max_cnt"),
).collect()[0]
print(f"   docID  [{stats['min_doc']}, {stats['max_doc']}]  expected [1, {D_kos}]")
print(f"   wordID [{stats['min_word']}, {stats['max_word']}]  expected [1, {W_kos}]")
print(f"   count  [{stats['min_cnt']}, {stats['max_cnt']}]")
assert stats["min_doc"] == 1 and stats["max_doc"] == D_kos
assert stats["min_cnt"] >= 1
print("   ✓ Assertions passed")

print("\n3. Duplicate Check:")
distinct_pairs = kos_df.select("docID", "wordID").distinct().count()
dups = rb_kos - distinct_pairs
print(f"   {'✓ No duplicates' if dups==0 else f'⚠ {dups} duplicates'}  [{distinct_pairs:,} unique pairs]")

print("\n4. Vocabulary Integrity:")
orphans = kos_df.select("wordID").distinct() \
    .subtract(kos_vocab_df.select("wordID").distinct()).count()
print(f"   {'✓ All wordIDs in vocab' if orphans==0 else f'⚠ {orphans} orphan wordIDs'}")

DATA VALIDATION — KOS

1. Null Check:
   ✓ docID: 0 nulls
   ✓ wordID: 0 nulls
   ✓ count: 0 nulls

2. Value Range Check:
   docID  [1, 3430]  expected [1, 3430]
   wordID [1, 6906]  expected [1, 6906]
   count  [1, 43]
   ✓ Assertions passed

3. Duplicate Check:
   ✓ No duplicates  [353,160 unique pairs]

4. Vocabulary Integrity:
   ✓ All wordIDs in vocab


In [11]:
# ── Cell 10: EDA Stats ───────────────────────────────────────
print("KOS raw statistics:")
kos_df.describe().show()

words_per_doc = kos_df.groupBy("docID").agg(
    F.sum("count").alias("total_words"),
    F.count("wordID").alias("unique_words")
)
docs_per_word = kos_df.groupBy("wordID").agg(
    F.count("docID").alias("doc_frequency"),
    F.sum("count").alias("total_occurrences")
)
print("Per-document stats:"); words_per_doc.describe().show()
print("Per-word stats:");     docs_per_word.describe().show()

KOS raw statistics:
+-------+------------------+------------------+------------------+
|summary|             docID|            wordID|             count|
+-------+------------------+------------------+------------------+
|  count|            353160|            353160|            353160|
|   mean|1713.6018405255409| 3545.417887076679| 1.324368558160607|
| stddev|  972.415220789158|1977.0697363562622|1.0393593058277146|
|    min|                 1|                 1|                 1|
|    max|              3430|              6906|                43|
+-------+------------------+------------------+------------------+

Per-document stats:
+-------+----------------+-----------------+------------------+
|summary|           docID|      total_words|      unique_words|
+-------+----------------+-----------------+------------------+
|  count|            3430|             3430|              3430|
|   mean|          1715.5|136.3597667638484|102.96209912536443|
| stddev|990.300038708808|96.8065661

In [12]:
# ── Cell 11: Save stats via PyArrow (avoids Spark write issues) ──
# Use PyArrow to save aggregated stats too — same safe pattern as raw data

def spark_df_to_parquet(spark_df, path):
    """Collect Spark DF to pandas then save via PyArrow — avoids HADOOP_HOME write issues."""
    pdf = spark_df.toPandas()
    pq.write_table(pa.Table.from_pandas(pdf, preserve_index=False), path, compression="snappy")
    print(f"  ✓ Saved {len(pdf):,} rows → {path}")

spark_df_to_parquet(words_per_doc, os.path.join(PAR_DIR, "kos_doc_stats.parquet"))
spark_df_to_parquet(docs_per_word, os.path.join(PAR_DIR, "kos_word_stats.parquet"))

  ✓ Saved 3,430 rows → ./data/parquet\kos_doc_stats.parquet
  ✓ Saved 6,906 rows → ./data/parquet\kos_word_stats.parquet


In [13]:
# ── Cell 12: Storage comparison & Summary ────────────────────
raw_mb     = os.path.getsize(kos_path) / 1024 / 1024
parquet_mb = os.path.getsize(kos_parquet_path) / 1024 / 1024

print(f"Storage — KOS docword:")
print(f"  Raw text : {raw_mb:.2f} MB")
print(f"  Parquet  : {parquet_mb:.2f} MB  ({raw_mb/parquet_mb:.1f}× smaller)")

print("\n" + "="*55)
print("INGESTION SUMMARY")
print("="*55)
print(f"  KOS  : {rb_kos:,} triples | {D_kos:,} docs | {W_kos:,} vocab")
print(f"  NIPS : {rb_nips:,} triples | {D_nips:,} docs | {W_nips:,} vocab")
print(f"  Write method : pandas → PyArrow → Parquet (bypasses HADOOP_HOME)")
print(f"  Read  method : Spark JVM Parquet reader (no Python workers)")
print(f"  Spark UI     : {spark.sparkContext.uiWebUrl}")
print("\nNotebook 1 complete ✓")

Storage — KOS docword:
  Raw text : 3.89 MB
  Parquet  : 0.85 MB  (4.6× smaller)

INGESTION SUMMARY
  KOS  : 353,160 triples | 3,430 docs | 6,906 vocab
  NIPS : 746,316 triples | 1,500 docs | 12,419 vocab
  Write method : pandas → PyArrow → Parquet (bypasses HADOOP_HOME)
  Read  method : Spark JVM Parquet reader (no Python workers)
  Spark UI     : http://DESKTOP-6826D4K:4043

Notebook 1 complete ✓
